## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-21-19-48-53 +0000,startribune,The Twin Cities has officially gone later than...,https://www.startribune.com/the-twin-cities-ha...
1,2026-11-21-19-45-46 +0000,nypost,Pelosi breaks vow to never utter ‘curse word’ ...,https://nypost.com/2026/11/21/us-news/pelosi-d...
2,2026-11-21-19-45-07 +0000,nypost,Alarm raised after 11 NSA workers commit suici...,https://nypost.com/2026/11/21/us-news/suicides...
3,2026-11-21-19-43-32 +0000,nypost,3.3-magnitude earthquake strikes off LA coast,https://nypost.com/2026/11/21/us-news/3-3-magn...
4,2026-11-21-19-43-22 +0000,chitri,Donald Trump vows to work with Democrats to ca...,https://www.chicagotribune.com/2026/11/21/trum...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2797/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
16,trump,36
137,california,17
462,new,13
488,he,12
67,top,11


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
179,2026-11-21-01-03-00 +0000,wsj,During President Trump’s White House briefing ...,https://www.wsj.com/politics/elections/how-a-w...,86
263,2026-11-20-21-35-47 +0000,cbc,Trump threatens to penalize U.S. tech companie...,https://www.cbc.ca/news/world/ai-data-centres-...,83
64,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...,80
252,2026-11-20-21-58-36 +0000,latimes,Trump says he'll penalize U.S. companies that ...,https://www.latimes.com/politics/story/2026-11...,78
58,2026-11-21-16-32-51 +0000,nypost,Trump defends Stephen Miller after Hakeem Jeff...,https://nypost.com/2026/11/21/us-news/trump-de...,74


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
179,86,2026-11-21-01-03-00 +0000,wsj,During President Trump’s White House briefing ...,https://www.wsj.com/politics/elections/how-a-w...
145,59,2026-11-21-09-53-45 +0000,nypost,FBI’s ‘Most Wanted’ cartel kingpin ‘Chapo Isid...,https://nypost.com/2026/11/21/world-news/fbis-...
16,50,2026-11-21-19-08-58 +0000,nypost,SpaceX shuttle to splash down off California c...,https://nypost.com/2026/11/21/us-news/spacex-d...
263,47,2026-11-20-21-35-47 +0000,cbc,Trump threatens to penalize U.S. tech companie...,https://www.cbc.ca/news/world/ai-data-centres-...
298,38,2026-11-20-20-12-57 +0000,nypost,Should Jensen Huang return as Nvidia CEO? Many...,https://nypost.com/2026/11/20/opinion/should-j...
238,38,2026-11-20-22-14-25 +0000,latimes,"Armed with guns and sword, owner of squatter r...",https://www.latimes.com/california/story/2026-...
58,38,2026-11-21-16-32-51 +0000,nypost,Trump defends Stephen Miller after Hakeem Jeff...,https://nypost.com/2026/11/21/us-news/trump-de...
64,36,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...
283,35,2026-11-20-20-41-00 +0000,wsj,Take-Two’s ‘Grand Theft Auto VI’ Hits Nearly $...,https://www.wsj.com/business/media/take-twos-g...
14,33,2026-11-21-19-13-56 +0000,nypost,"2 dead, 60 injured in Russian strike on Zelens...",https://nypost.com/2026/11/21/world-news/2-dea...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
